# NS12 Tutorial — Communities: Definitions, Partitions, and Network Partitioning

**Lecture:** NS12 — Communities  
**Reading:** Ch. 6 [ns1]; Schaub et al., *The many facets of community detection in complex networks* (2017); Fortunato & Hric, *Community detection in networks: a user guide* (2016).

## Why this notebook exists

Before we can *detect* communities (NB 13) or *evaluate* them against ground truth (NB 14), we need
to pin down what a community *is*. This is surprisingly subtle: there is **no single accepted
definition**, only a family of criteria that agree in clean cases and disagree at the boundaries.
This notebook builds the vocabulary carefully, starting from the ground up.

## Learning goals

- Build intuition for why **community structure** is ubiquitous in real networks.
- Formalise a community via its **internal/external degree**, **cohesion**, and **separation**.
- Distinguish **strong** vs **weak** communities in both the *stringent* and *less-stringent* forms,
  and understand *why* the less-stringent forms exist.
- Appreciate the size of the **partition space** (Bell numbers) and the even larger space of **covers**.
- Run **Kernighan–Lin** bisection step-by-step and understand why *graph partitioning ≠ community detection*.
- Use **hierarchical clustering** with **structural equivalence** to produce **dendrograms**;
  compare **single / complete / average linkage**.
- Learn the vocabulary — *cohesion, separation, spanning links, cut size, stringent / less-stringent,
  cover, dendrogram, structural equivalence, agglomerative / divisive* — used in NB 13 and NB 14.

## Outline

1. Motivation — a gallery of real community structure, plus a preview on the Karate Club.
2. Community variables — internal/external degree, community degree, internal density.
3. Strong vs weak communities — stringent and less-stringent forms.
4. The partition space — Bell numbers, covers, hierarchies.
5. Network partitioning — Kernighan–Lin, worked step by step.
6. Hierarchical clustering — structural equivalence, linkage, dendrograms.
7. Exercises.
8. Takeaway and bridge to NS 13 / NS 14.

## Datasets used throughout notebooks 12–14

- **Zachary's Karate Club** — 34 nodes, 78 edges. Ground-truth split: *Mr. Hi* vs *Officer*.
- **Les Misérables** character co-occurrence — 77 characters, weighted by number of shared chapters.
- Several **toy hand-built graphs** (two triangles, barbell, two-cliques-with-bridge, nested cliques)
  used to build intuition *before* turning to real data.

## Companion notebooks — what is covered *elsewhere*

- **NB 13** — Girvan–Newman, modularity, greedy Newman, Louvain. We *define* modularity here only in
  passing when evaluating a hierarchical cut; the full treatment is in NB 13.
- **NB 14** — Label propagation, NMI / ARI, LFR benchmarks. We *mention* NMI at the end; again, the
  full treatment is in NB 14.

In [ ]:
from netsci_utils import *

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
from math import comb
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

set_seeds()

# --- Two real datasets used consistently across notebooks 12–14 ---
G_karate = nx.karate_club_graph()
G_lesmis = nx.les_miserables_graph()

# Integer encoding of the Karate ground-truth split (also used in NB 13 and NB 14).
# We explicitly fix the convention: 0 = Mr. Hi's faction, 1 = Officer's faction.
karate_truth = {n: (0 if d['club'] == 'Mr. Hi' else 1)
                for n, d in G_karate.nodes(data=True)}

print(f'Karate Club    : {G_karate.number_of_nodes()} nodes, {G_karate.number_of_edges()} edges')
print(f'Les Misérables : {G_lesmis.number_of_nodes()} nodes, {G_lesmis.number_of_edges()} edges (weighted)')

# Cache a shared spring layout so that every Karate figure in the notebook is visually consistent.
pos_karate = nx.spring_layout(G_karate, seed=RANDOM_SEED)

---
## 1. Motivation — communities are everywhere

### 1.1 A gallery of real community structure

Look at almost any real network through the lens of *density* and you will find groups of nodes
that stick together much more than average:

- **Political polarisation on Twitter / X.** Retweet networks of political activity famously split
  into two nearly-disjoint clusters, one left-leaning and one right-leaning, connected by just a
  thin band of *spanning links* (journalists, bots, a few celebrities). Adamic & Glance (2005) and
  Conover et al. (2011) gave the canonical illustration.
- **Protein–protein interaction networks.** Functional modules (ribosome, proteasome, spliceosome)
  appear as dense subgraphs. Community detection is one of the standard tools used to *discover*
  previously unknown modules and *predict* the function of uncharacterised proteins.
- **The Web.** Pages about the same topic point at each other far more often than at unrelated
  topics; communities align with *thematic clusters* (finance sites, cooking blogs, academic homepages).
- **Friendship and collaboration networks.** Schoolmates, coworkers, sports clubs, research groups
  — social networks are almost by definition collections of overlapping circles.

### 1.2 Why study communities?

- **Uncover organisation.** The partition reveals the modular backbone of the system.
- **Infer node labels.** Unknown attributes can be guessed from the community a node sits in.
- **Recommend.** Suggest friends, items, pages inside a user's community.
- **Predict missing links.** Edges are more likely inside a community than between two.
- **Summarise large networks.** Replace a noisy node-level view with a coarse inter-community one.

### 1.3 Two features that will drive every definition

- **Cohesion.** Many *internal links* — the nodes stick together.
- **Separation.** Few *external links* — the community is only loosely connected to the outside.

The few edges that *do* cross a community boundary will be called **spanning links** (the term used
by Girvan–Newman in NB 13 when it iteratively removes them).

### 1.4 Preview — Karate Club, ground truth vs. a Louvain partition

Before we get technical, here is the punchline. On the left we colour the Karate nodes by the
**historical ground truth** (Mr. Hi vs. Officer). On the right we run a modern modularity-maximising
algorithm (**Louvain**, covered properly in NB 13) and colour by its output. The two agree
remarkably well, which is essentially the reason Karate has become the *Drosophila* of community
detection.

In [ ]:
# Ground-truth colouring
colors_truth = [CATEGORY_PALETTE['red'] if karate_truth[n] == 0 else CATEGORY_PALETTE['blue']
                for n in G_karate.nodes()]

# Louvain partition (full treatment in NB 13 — here only as a hook)
louvain_parts = nx.community.louvain_communities(G_karate, seed=RANDOM_SEED)
louvain_label = {n: i for i, C in enumerate(louvain_parts) for n in C}
palette_list = list(CATEGORY_PALETTE.values())
colors_louvain = [palette_list[louvain_label[n] % len(palette_list)] for n in G_karate.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
draw_graph(G_karate, pos=pos_karate, ax=axes[0], node_color=colors_truth, node_size=400, font_size=8)
axes[0].set_title("Ground truth (Mr. Hi vs. Officer)")
draw_graph(G_karate, pos=pos_karate, ax=axes[1], node_color=colors_louvain, node_size=400, font_size=8)
axes[1].set_title(f"Louvain partition — {len(louvain_parts)} communities (preview of NB 13)")
plt.show()

print(f'Louvain found {len(louvain_parts)} communities of sizes {sorted(len(C) for C in louvain_parts)}.')

**Interpretation.** The historical split is essentially a **2-community partition**, while Louvain
detects *finer* structure (typically 3–4 communities on this graph). Neither is "wrong" — they
describe the network at different *resolutions*. This tension between resolutions is a recurring
theme of the field and shows up again in NB 13 when we discuss the *resolution limit* of modularity.

### 1.5 From intuition to vocabulary

The gallery and the Karate preview make *communities* feel intuitive — we "see" them.
To **measure** them, we need precise vocabulary. The next section defines the raw variables
(internal / external degree, community degree, internal link density) that every later
algorithm will build on. Think of Section 2 as the *unit-conversion chart* of community
detection: every formula we will meet decomposes into these five numbers.

---
## 2. Community variables — deep dive

Fix a community $C \subseteq V$ and a node $v \in C$. Four quantities will drive every definition in
the notebook:

- **Internal degree** $k_v^{int}$: number of neighbours of $v$ that lie *inside* $C$ (excluding $v$
  itself so self-loops do not inflate the count).
- **External degree** $k_v^{ext}$: number of neighbours of $v$ that lie *outside* $C$.
- **Community degree** of $C$: $k_C = \sum_{v \in C} k_v$, the sum of the *total* degrees of the nodes in $C$.
- **Internal link density**:
$$\delta_C^{int} \;=\; \frac{L_C}{L_C^{max}} \;=\; \frac{L_C}{\binom{N_C}{2}} \;=\; \frac{2 L_C}{N_C(N_C-1)}$$
where $L_C$ is the number of internal links in $C$ and $N_C = |C|$.

A good community should have **$\delta_C^{int}$ close to 1** (cohesion) and most of its $k_C$ mass
in internal links (separation).

### 2.1 Toy warm-up — two triangles joined by one edge

Before we compute anything on Karate, let us build the *simplest non-trivial example*: two
triangles $\{a,b,c\}$ and $\{d,e,f\}$ connected by a single edge $c$–$d$. With six nodes and seven
edges we can check every number by hand.

In [ ]:
def two_triangles_bridge():
    G = nx.Graph()
    G.add_edges_from([('a','b'),('b','c'),('a','c'),   # triangle 1
                      ('d','e'),('e','f'),('d','f'),   # triangle 2
                      ('c','d')])                       # single bridge
    return G

G_toy = two_triangles_bridge()
pos_toy = {'a':(-2,1),'b':(-2,-1),'c':(-1,0),'d':(1,0),'e':(2,1),'f':(2,-1)}
C_left  = ['a','b','c']
C_right = ['d','e','f']

node_colors = [CATEGORY_PALETTE['red'] if n in C_left else CATEGORY_PALETTE['blue']
               for n in G_toy.nodes()]
plt.figure(figsize=(7, 3.5))
draw_graph(G_toy, pos=pos_toy, node_color=node_colors, node_size=800)
plt.title('Toy graph — two triangles connected by a single spanning link')
plt.show()

In [ ]:
def internal_external_degree(G, v, community):
    """Return (k_int, k_ext) for v w.r.t. *community*.

    We explicitly exclude v itself from C and from the neighbour set so the function is robust
    against self-loops and the degenerate case v in community.
    """
    C = set(community) - {v}
    nbrs = set(G.neighbors(v)) - {v}
    return len(nbrs & C), len(nbrs - C)


def community_degree(G, community):
    """Sum of the TOTAL degrees of the nodes in *community*."""
    return sum(dict(G.degree(community)).values())


def internal_link_density(G, community):
    H = G.subgraph(community)
    n = H.number_of_nodes()
    if n < 2:
        return 0.0
    return 2 * H.number_of_edges() / (n * (n - 1))


# Per-node table for the LEFT triangle
rows = []
for v in C_left:
    kin, kout = internal_external_degree(G_toy, v, C_left)
    rows.append({'node': v, 'k_int': kin, 'k_ext': kout, 'degree': G_toy.degree(v)})
display(pd.DataFrame(rows))

print(f"community degree k_C = {community_degree(G_toy, C_left)}")
print(f"internal links L_C  = {G_toy.subgraph(C_left).number_of_edges()}")
print(f"internal density    = {internal_link_density(G_toy, C_left):.3f}")

**Reading the table.** Nodes $a$ and $b$ are *pure insiders* (two neighbours inside $C$, none
outside). Node $c$ is a *boundary* node: it still has two internal neighbours but it also *exports*
one edge to the other community. The internal density is $\delta = 2 \cdot 3 / (3 \cdot 2) = 1$ —
the subgraph is a clique.

### Variable-by-variable intuition

| Variable | Low values mean… | High values mean… | Diagnoses |
|---|---|---|---|
| $k_v^{int}$ | $v$ is loosely integrated inside $C$ | $v$ is a *core* member of $C$ | cohesion |
| $k_v^{ext}$ | $v$ is well-insulated from the outside | $v$ is a *boundary* / *broker* | separation |
| $k_C$ | $C$ is small and/or low-degree | $C$ aggregates a lot of the graph's volume | size + importance |
| $\delta_C^{int}$ | $C$ is *sparse internally* (suspicious) | $C$ is *clique-like* | cohesion (normalised) |

### 2.2 Karate Club — every variable, every community

Let us now repeat the calculation on the real ground-truth split of the Karate Club.

In [ ]:
C_hi     = [n for n, c in karate_truth.items() if c == 0]
C_office = [n for n, c in karate_truth.items() if c == 1]
partition_truth = [C_hi, C_office]

summary = []
for name, C in [('Mr. Hi', C_hi), ('Officer', C_office)]:
    summary.append({
        'community': name,
        '|C|': len(C),
        'L_C (internal)': G_karate.subgraph(C).number_of_edges(),
        'k_C (community degree)': community_degree(G_karate, C),
        'delta_int': round(internal_link_density(G_karate, C), 3),
    })
display(pd.DataFrame(summary))

In [ ]:
# Random baseline for internal density: draw a random subset of the same size and measure delta_int
rng = np.random.default_rng(RANDOM_SEED)
nodes_all = list(G_karate.nodes())

rows = []
for name, C in [('Mr. Hi', C_hi), ('Officer', C_office)]:
    observed = internal_link_density(G_karate, C)
    baseline = [internal_link_density(G_karate, rng.choice(nodes_all, size=len(C), replace=False).tolist())
                for _ in range(500)]
    rows.append({
        'community': name,
        'observed delta_int': round(observed, 3),
        'random mean delta_int': round(float(np.mean(baseline)), 3),
        'random 95th pct': round(float(np.percentile(baseline, 95)), 3),
        'z-score': round((observed - np.mean(baseline)) / np.std(baseline), 2),
    })
display(pd.DataFrame(rows))

**Interpretation.** Both factions are denser than random subsets of equal size by several standard
deviations. This is exactly what *cohesion* means quantitatively: $\delta_C^{int}$ is far above what
you would expect if the same number of nodes were sampled at random.

In [ ]:
# Histograms: internal and external degree distributions inside Mr. Hi's faction
kin_hi  = [internal_external_degree(G_karate, v, C_hi)[0] for v in C_hi]
kout_hi = [internal_external_degree(G_karate, v, C_hi)[1] for v in C_hi]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(kin_hi,  bins=range(0, max(kin_hi)+2),  color=CATEGORY_PALETTE['red'], edgecolor='white')
style_axis(axes[0], title="Mr. Hi's faction — internal degree $k_v^{int}$",
           xlabel='$k_v^{int}$', ylabel='count')
axes[1].hist(kout_hi, bins=range(0, max(kout_hi)+2), color=CATEGORY_PALETTE['orange'], edgecolor='white')
style_axis(axes[1], title="Mr. Hi's faction — external degree $k_v^{ext}$",
           xlabel='$k_v^{ext}$', ylabel='count')
plt.show()

print(f"mean k_int = {np.mean(kin_hi):.2f},  mean k_ext = {np.mean(kout_hi):.2f}")

**Interpretation.** The internal degree distribution is shifted clearly to the right of the
external one — on average, members of Mr. Hi's faction have *more* ties inside the faction than
outside. That imbalance is exactly what the *weak community* definition in Section 3 is going to
formalise.

---
## 3. Strong vs weak communities

The folk idea — *"more internal links than external links"* — can be formalised in two levels of
strictness.

### 3.1 Stringent forms

- **Strong community (stringent).** Every node satisfies the inequality:
  $$ k_v^{int} > k_v^{ext}, \qquad \forall\, v \in C. $$
- **Weak community (stringent).** The inequality holds *only on the sum*:
  $$ \sum_{v \in C} k_v^{int} \;>\; \sum_{v \in C} k_v^{ext}. $$

A *strong* community is automatically *weak* (per-node beats sum). The converse is not true.

### 3.2 Why “less-stringent”

The stringent definitions compare $C$ against *the whole rest of the network*. But a node $v$ does
not really care whether its neighbours are scattered across the rest of the graph — it cares
whether any *single alternative community* could steal it. The **less-stringent** versions replace
*“external”* with *“neighbours in the most attractive single other community”*:

- **Strong (less stringent).** For every $v \in C$, $k_v^{int} > k_v^{D}$ for every other community $D$.
- **Weak (less stringent).** $\sum_v k_v^{int} > \sum_v k_v^{D}$ for every other community $D$.

This matters when a network has *many* communities: a node can have most of its edges outside $C$
when we lump all “outers” together, yet still have a plurality of neighbours in $C$ when we compare
against *one* alternative at a time.

### 3.3 Pedagogic mini-graph — *weak but not strong*

Construct a tiny graph with two candidate communities. The left community is **weak-but-not-strong**:
the *sum* of internal degrees beats the sum of external degrees, but *one node* has more external
than internal neighbours.

In [ ]:
def weak_not_strong_demo():
    G = nx.Graph()
    # Left community: clique on {1,2,3,4} + node 5 with 1 internal, 2 external ties
    G.add_edges_from([(1,2),(1,3),(1,4),(2,3),(2,4),(3,4),(4,5)])
    # Right community: clique on {6,7,8}
    G.add_edges_from([(6,7),(6,8),(7,8)])
    # Node 5 has TWO external ties → strongly pulled outside, violating per-node strong
    G.add_edges_from([(5,6),(5,7)])
    return G

G_wns = weak_not_strong_demo()
CL = [1,2,3,4,5]; CR = [6,7,8]
pos_wns = nx.spring_layout(G_wns, seed=RANDOM_SEED)
colors_wns = [CATEGORY_PALETTE['red'] if n in CL else CATEGORY_PALETTE['blue'] for n in G_wns.nodes()]

plt.figure(figsize=(7, 4))
draw_graph(G_wns, pos=pos_wns, node_color=colors_wns, node_size=700)
plt.title('Weak-but-not-strong demo — node 5 has more ties outside than inside $C_L$')
plt.show()

rows = []
for v in CL:
    kin, kout = internal_external_degree(G_wns, v, CL)
    rows.append({'node': v, 'k_int': kin, 'k_ext': kout, 'strong violated?': kin <= kout})
display(pd.DataFrame(rows))
print(f"Sum k_int = {sum(internal_external_degree(G_wns, v, CL)[0] for v in CL)},"
      f"  Sum k_ext = {sum(internal_external_degree(G_wns, v, CL)[1] for v in CL)}")

**Reading.** Node 5 fails the per-node inequality (1 inside, 2 outside), so $C_L$ is **not strong**.
But $\sum k_v^{int} = 9$ and $\sum k_v^{ext} = 2$ inside $C_L$, so $C_L$ is **weak**. This is the
purest illustration of the difference between the two definitions.

### 3.4 Stringent vs less-stringent — when a *third* community changes the answer

Here is the conceptual move. Build a graph with three communities. A node in $C_1$ has four
neighbours: 2 in $C_1$, 1 in $C_2$, 1 in $C_3$. By the *stringent* strong criterion it fails
($k_v^{int} = 2$ vs. $k_v^{ext} = 2$), but by the *less-stringent* criterion it succeeds, because
no single alternative community has more than 1 of its neighbours.

In [ ]:
def three_communities_demo():
    G = nx.Graph()
    # Three triangles
    G.add_edges_from([(1,2),(1,3),(2,3)])          # C1
    G.add_edges_from([(4,5),(4,6),(5,6)])          # C2
    G.add_edges_from([(7,8),(7,9),(8,9)])          # C3
    # Cross-links: node 1 in C1 has one tie to C2 and one to C3
    G.add_edges_from([(1,4),(1,7)])
    return G

G3 = three_communities_demo()
C1, C2, C3 = [1,2,3], [4,5,6], [7,8,9]
parts3 = [C1, C2, C3]
pos3 = nx.spring_layout(G3, seed=RANDOM_SEED)
palette = [CATEGORY_PALETTE['red'], CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['green']]
colors3 = [palette[next(i for i,C in enumerate(parts3) if n in C)] for n in G3.nodes()]

plt.figure(figsize=(7, 4.5))
draw_graph(G3, pos=pos3, node_color=colors3, node_size=700)
plt.title('Three-community demo — node 1 has 2 internal, 1+1 split external')
plt.show()

v = 1
kin, kout = internal_external_degree(G3, v, C1)
k_to_C2 = len(set(G3.neighbors(v)) & set(C2))
k_to_C3 = len(set(G3.neighbors(v)) & set(C3))
print(f'Node {v}: k_int={kin}, k_ext={kout}, k_to_C2={k_to_C2}, k_to_C3={k_to_C3}')
print(f'Stringent strong at node {v}?     {kin > kout}')
print(f'Less-stringent strong at node {v}? {kin > max(k_to_C2, k_to_C3)}')

**Takeaway.** When communities are *many*, the stringent criterion is overly pessimistic because
it treats the aggregate outside as a single adversary. The less-stringent criterion is the one most
widely used in practice; it is the definition most often meant when a paper says “strong” or “weak”
community without further qualification.

In [ ]:
def is_strong_community(G, community):
    C = set(community)
    return all(internal_external_degree(G, v, C)[0] > internal_external_degree(G, v, C)[1]
               for v in C)


def is_weak_community(G, community):
    C = set(community)
    kin = sum(internal_external_degree(G, v, C)[0] for v in C)
    kout = sum(internal_external_degree(G, v, C)[1] for v in C)
    return kin > kout


def is_strong_less_stringent(G, community, partition):
    C = set(community)
    others = [set(D) for D in partition if set(D) != C]
    for v in C:
        kin = len(set(G.neighbors(v)) & (C - {v}))
        max_other = max((len(set(G.neighbors(v)) & D) for D in others), default=0)
        if kin <= max_other:
            return False
    return True


def is_weak_less_stringent(G, community, partition):
    C = set(community)
    others = [set(D) for D in partition if set(D) != C]
    kin_sum = sum(len(set(G.neighbors(v)) & (C - {v})) for v in C)
    if not others:
        return True
    max_other_sum = max(sum(len(set(G.neighbors(v)) & D) for v in C) for D in others)
    return kin_sum > max_other_sum


# Apply all four tests to the Karate ground-truth factions
rows = []
for name, C in [('Mr. Hi', C_hi), ('Officer', C_office)]:
    rows.append({
        'community': name,
        'strong (stringent)': is_strong_community(G_karate, C),
        'weak (stringent)':   is_weak_community(G_karate, C),
        'strong (less str.)': is_strong_less_stringent(G_karate, C, partition_truth),
        'weak (less str.)':   is_weak_less_stringent(G_karate, C, partition_truth),
    })
display(pd.DataFrame(rows))

In [ ]:
# Per-node int/ext degree table for Mr. Hi's faction, highlighting stringent-strong violators
rows = []
for v in sorted(C_hi):
    kin, kout = internal_external_degree(G_karate, v, C_hi)
    rows.append({'node': v, 'k_int': kin, 'k_ext': kout,
                 'violates stringent strong?': kin <= kout})

df = pd.DataFrame(rows)
display(df)

violators = df.loc[df['violates stringent strong?'], 'node'].tolist()
print(f'Violators (stringent strong) inside Mr. Hi: {violators}')

In [ ]:
# Also apply the four tests to the Louvain partition we computed earlier
rows = []
for i, C in enumerate(louvain_parts):
    rows.append({
        'louvain community': f'C{i}',
        '|C|': len(C),
        'strong (stringent)': is_strong_community(G_karate, C),
        'weak (stringent)':   is_weak_community(G_karate, C),
        'strong (less str.)': is_strong_less_stringent(G_karate, C, louvain_parts),
        'weak (less str.)':   is_weak_less_stringent(G_karate, C, louvain_parts),
    })
display(pd.DataFrame(rows))

**Interpretation.** The ground-truth Karate factions are *weak* on the sum and *strong in the
less-stringent sense*, but a handful of peripheral nodes violate the **stringent** per-node
condition. Louvain's finer partition obeys the less-stringent conditions even more cleanly. This is
the standard picture: real communities almost never pass the stringent per-node criterion, which is
one of the main reasons the definition is rarely used without qualification.

**Concept check.** If every node in $C$ has $k_v^{int} = k_v^{ext}$ (strict equality), is $C$
strong? Weak? Stringent or less-stringent? *Answer: neither strong nor weak under the stringent
definitions, because the inequalities are strict. It can still be less-stringent strong if every
single alternative community would capture strictly fewer than $k_v^{int}$ neighbours.*

### 3.5 Why we now count partitions

Strong and weak are statements **about a single community**. But a real network has *many*
communities at once — and we have to assign *every* node to exactly one. That is a
**partition**. A natural next question is: *how many possible partitions exist on $N$
nodes*? The answer — the **Bell number** — grows faster than exponentially and immediately
tells us why exhaustive search is hopeless. Section 4 develops this argument and surveys
the richer structures (covers, hierarchies) that communities can also take.

---
## 4. Partitions, Bell numbers, covers, hierarchies

### 4.1 What is a partition?

A **partition** of the node set $V$ is a collection of *non-empty, pairwise disjoint* subsets whose
union is $V$. Every node belongs to *exactly one* community.

### 4.2 Enumerating all partitions of a tiny set

For three elements $\{a,b,c\}$ there are exactly $B_3 = 5$ partitions:

1. $\{\{a,b,c\}\}$ — one big community.
2. $\{\{a,b\},\{c\}\}$.
3. $\{\{a,c\},\{b\}\}$.
4. $\{\{b,c\},\{a\}\}$.
5. $\{\{a\},\{b\},\{c\}\}$ — singleton partition.

Any community-detection algorithm chooses *one* element from this combinatorial zoo, guided by some
objective (modularity, cut size, likelihood, …).

In [ ]:
def all_partitions(collection):
    """Yield all set partitions of *collection* (Bell-number many)."""
    collection = list(collection)
    if len(collection) == 1:
        yield [collection]; return
    first = collection[0]
    for smaller in all_partitions(collection[1:]):
        for i, subset in enumerate(smaller):
            yield smaller[:i] + [[first] + subset] + smaller[i+1:]
        yield [[first]] + smaller

for i, P in enumerate(all_partitions(['a','b','c']), 1):
    print(f'{i}. {P}')

### 4.3 Bell numbers — partitions explode

The number of partitions of an $n$-element set is the **Bell number** $B_n$, given by the
recurrence

$$B_{n+1} = \sum_{k=0}^{n} \binom{n}{k} B_k, \qquad B_0 = 1.$$

$B_n$ grows **faster than exponentially**. For $n=20$, $B_{20} \approx 5.2\times 10^{13}$; for
$n=50$, $B_{50} \approx 1.9\times 10^{47}$. Exhaustive search is **computationally impossible**
for anything beyond toy instances — hence the need for heuristics in NB 13 and NB 14.

In [ ]:
def bell_numbers(N):
    B = [1]
    for n in range(1, N + 1):
        B.append(sum(comb(n-1, k) * B[k] for k in range(n)))
    return B

B = bell_numbers(30)
for n in [0, 1, 2, 3, 5, 10, 15, 20, 25, 30]:
    print(f'B_{n:<2d} = {B[n]:,}')

In [ ]:
# Pedagogical figure: partition space vs. graph space
# Number of possible simple graphs on n labelled nodes = 2^(n*(n-1)/2).
# Compare growth: both are enormous but the curves are instructive.
ns = np.arange(1, 31)
bells = np.array([B[n] for n in ns], dtype=float)
graphs = 2.0 ** (ns * (ns - 1) / 2)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(ns, bells,  marker='o', color=CATEGORY_PALETTE['blue'],   label='$B_n$ — partitions of $n$ nodes')
ax.semilogy(ns, graphs, marker='s', color=CATEGORY_PALETTE['orange'], label='$2^{n(n-1)/2}$ — labelled simple graphs on $n$ nodes')
style_axis(ax, title='Partition space explodes super-exponentially — but graph space explodes faster still',
           xlabel='$n$', ylabel='count (log scale)', legend=True)
plt.show()

### 4.4 Covers — overlapping communities

Partitions forbid overlap, but real communities overlap all the time — think of a researcher with
joint appointments, or a person belonging both to a family and a workplace. A **cover** is a
collection of node sets whose union is $V$, without the disjointness requirement. The number of
possible covers of an $n$-element set is $2^{2^n - 1}$, *much* larger than $B_n$.

Here is a toy illustration: a node that sits in two communities at once.

In [ ]:
def overlapping_cover_demo():
    G = nx.Graph()
    # Community A: {1,2,3,4}, community B: {4,5,6,7}. Node 4 is shared.
    G.add_edges_from([(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)])  # A: clique
    G.add_edges_from([(4,5),(4,6),(4,7),(5,6),(5,7),(6,7)])  # B: clique
    return G

G_cov = overlapping_cover_demo()
pos_cov = nx.spring_layout(G_cov, seed=RANDOM_SEED)
A = {1,2,3,4}; B = {4,5,6,7}
colors_cov = []
for n in G_cov.nodes():
    if n in A and n in B:
        colors_cov.append(CATEGORY_PALETTE['purple'])  # shared
    elif n in A:
        colors_cov.append(CATEGORY_PALETTE['red'])
    else:
        colors_cov.append(CATEGORY_PALETTE['blue'])

plt.figure(figsize=(7, 4))
draw_graph(G_cov, pos=pos_cov, node_color=colors_cov, node_size=700)
plt.title('Overlapping cover — node 4 belongs to both communities (purple)')
plt.show()

Covers are the proper framework for algorithms such as **clique percolation** and **BIGCLAM**.
Since their space is astronomically larger than the partition space, the search problem is even
harder; we do not pursue it in NB 13 but the vocabulary is worth knowing.

### 4.5 Hierarchies — communities inside communities

Many real systems are **hierarchical**: a company decomposes into divisions, divisions into teams,
teams into individuals. Formally, a hierarchy is a *tree* of partitions, where each level refines
the previous one. The dendrograms of Section 6 encode exactly this structure.

In [ ]:
def nested_cliques(clique_size=5, n_cliques=4, bridge_probability=1.0, seed=RANDOM_SEED):
    """Four cliques of size `clique_size`, arranged 2x2; connect each pair of cliques with one bridge."""
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    blocks = []
    for b in range(n_cliques):
        nodes = list(range(b*clique_size, (b+1)*clique_size))
        G.add_nodes_from(nodes)
        for u, v in combinations(nodes, 2):
            G.add_edge(u, v)
        blocks.append(nodes)
    # Bridges: (0<->1), (2<->3) at the fine level; (0<->2), (1<->3) across the coarse level
    pairs_fine   = [(0,1),(2,3)]
    pairs_coarse = [(0,2),(1,3)]
    for i,j in pairs_fine + pairs_coarse:
        u = rng.choice(blocks[i]); v = rng.choice(blocks[j])
        G.add_edge(int(u), int(v))
    level1 = [blocks[0]+blocks[1], blocks[2]+blocks[3]]  # coarse: 2 super-communities
    level2 = blocks                                        # fine: 4 cliques
    return G, level1, level2

G_nest, lvl1, lvl2 = nested_cliques()
pos_nest = nx.spring_layout(G_nest, seed=RANDOM_SEED)

col_l1 = [CATEGORY_PALETTE['red']  if n in lvl1[0] else CATEGORY_PALETTE['blue']  for n in G_nest.nodes()]
pal    = [CATEGORY_PALETTE['red'], CATEGORY_PALETTE['orange'], CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['green']]
col_l2 = [pal[next(i for i,C in enumerate(lvl2) if n in C)] for n in G_nest.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
draw_graph(G_nest, pos=pos_nest, ax=axes[0], node_color=col_l1, node_size=350, font_size=7)
axes[0].set_title('Coarse level — 2 super-communities')
draw_graph(G_nest, pos=pos_nest, ax=axes[1], node_color=col_l2, node_size=350, font_size=7)
axes[1].set_title('Fine level — 4 cliques')
plt.show()

### 4.5 A detour before community detection

Section 5 takes a short *detour* from community **detection** to **graph partitioning** —
a conceptually different problem where the number of groups (and often their size) is
**given in advance**. Why spend time on it? Two reasons:

1. It motivates the **cut size** as a fundamental structural quantity (we will re-use it
   as a baseline in NB 13).
2. Its algorithm — **Kernighan–Lin** — is a classical, still-used tool, and the
   gain-swap-lock logic it uses reappears in Louvain's Phase 1 (NB 13).

After the detour we will return to true community detection in NB 13.

---
## 5. Network partitioning — the Kernighan–Lin algorithm

### 5.1 Not the same problem as community detection

**Network partitioning** fixes in advance *how many groups* and *how large each group* should be,
then minimises the **cut size** (number of edges between groups). The cross-group edges are exactly
the *spanning links* we met earlier.

- **Community detection.** Number and sizes of the communities emerge from the data.
- **Network partitioning.** Number and sizes are *inputs*; we only optimise the cut.

**Classical applications**

- *Electronic circuit design.* Place gates on two boards so as to minimise the *wires* crossing
  between boards; each board holds a fixed number of gates.
- *Parallel computing.* Assign tasks to $k$ processors so as to minimise *inter-processor
  communication*; each processor gets a fixed quota.

### 5.2 Kernighan–Lin, the bisection case

1. **Initialisation.** Pick a random bisection $A \cup B = V$ with $|A| = |B|$. Compute the cut size.
2. **Gain table.** For every pair $(a,b)$ with $a \in A$, $b \in B$, compute the *gain*
   $g_{ab}$ = reduction in cut size if we swap $a$ and $b$.
3. **Pass.** Repeatedly pick the pair with maximum gain, *swap them even if $g < 0$*, and
   **lock** the two nodes. Record the cumulative gain $G_k = g_1 + \cdots + g_k$.
4. **Commit.** Find $k^* = \arg\max_k G_k$. Apply swaps $1, \ldots, k^*$ (roll back the rest).
   If $G_{k^*} > 0$, unlock everything and start another pass. Otherwise stop.

The algorithm is **greedy** but accepts occasional negative-gain swaps, which lets it escape shallow
local minima. The quality of the result still depends sharply on the random initial bisection.

### 5.3 A small worked example — 8 nodes, step by step

We pick an 8-node graph with a natural *4+4* bisection, start from a *bad* random split, and watch
KL peel the cut apart.

In [ ]:
def small_kl_graph():
    G = nx.Graph()
    # Two 4-cliques, one bridge
    G.add_edges_from([(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)])
    G.add_edges_from([(5,6),(5,7),(5,8),(6,7),(6,8),(7,8)])
    G.add_edge(4, 5)
    return G

G8 = small_kl_graph()
pos8 = nx.spring_layout(G8, seed=RANDOM_SEED)


def cut_size_simple(G, A, B):
    """Number of edges crossing the bisection A | B."""
    A, B = set(A), set(B)
    return sum(1 for u, v in G.edges()
               if (u in A and v in B) or (u in B and v in A))


# Deliberately bad initial bisection
A0 = [1, 2, 5, 6]
B0 = [3, 4, 7, 8]
print(f'Initial cut size: {cut_size_simple(G8, A0, B0)}')

colors8 = [CATEGORY_PALETTE['red'] if n in A0 else CATEGORY_PALETTE['blue'] for n in G8.nodes()]
plt.figure(figsize=(7, 4))
draw_graph(G8, pos=pos8, node_color=colors8, node_size=700)
plt.title(f'KL demo — random initial bisection (cut = {cut_size_simple(G8, A0, B0)})')
plt.show()


In [ ]:
def kl_d_values(G, A, B):
    """External minus internal degree for every node: D_v = E_v - I_v."""
    A, B = set(A), set(B)
    D = {}
    for v in G.nodes():
        own = A if v in A else B
        other = B if v in A else A
        I = sum(1 for u in G.neighbors(v) if u in own)
        E = sum(1 for u in G.neighbors(v) if u in other)
        D[v] = E - I
    return D


def kl_gain_table(G, A, B):
    """Gain g_ab = D_a + D_b - 2*w(a,b)."""
    D = kl_d_values(G, A, B)
    rows = []
    for a in A:
        for b in B:
            w = 1 if G.has_edge(a, b) else 0
            rows.append({'a': a, 'b': b, 'D_a': D[a], 'D_b': D[b], 'w(a,b)': w,
                         'g = D_a+D_b-2w': D[a] + D[b] - 2*w})
    return pd.DataFrame(rows).sort_values('g = D_a+D_b-2w', ascending=False).reset_index(drop=True)

display(kl_gain_table(G8, A0, B0).head(10))

In [ ]:
def kl_one_pass(G, A, B):
    """Run ONE KL pass (locking, cumulative gain trace). Return trace and best prefix."""
    A, B = list(A), list(B)
    locked = set()
    swaps, gains = [], []
    while True:
        remA = [a for a in A if a not in locked]
        remB = [b for b in B if b not in locked]
        if not remA or not remB:
            break
        D = kl_d_values(G, A, B)
        best = None; best_g = -float('inf')
        for a in remA:
            for b in remB:
                w = 1 if G.has_edge(a, b) else 0
                g = D[a] + D[b] - 2*w
                if g > best_g:
                    best_g = g; best = (a, b)
        a, b = best
        A = [b if x == a else x for x in A]
        B = [a if x == b else x for x in B]
        locked.update({a, b})
        swaps.append((a, b)); gains.append(best_g)
    cumulative = np.cumsum(gains)
    k_star = int(np.argmax(cumulative))  # best prefix
    return swaps, gains, cumulative, k_star


swaps, gains, cumulative, k_star = kl_one_pass(G8, A0, B0)
trace = pd.DataFrame({'step': range(1, len(swaps)+1),
                      'swap': swaps,
                      'gain': gains,
                      'cumulative gain G_k': cumulative})
display(trace)
print(f'Best prefix: commit the first k* = {k_star+1} swap(s) (cumulative gain {cumulative[k_star]}).')

In [ ]:
# Apply the best prefix and visualise BEFORE / AFTER / TRACE
A_cur, B_cur = list(A0), list(B0)
for i in range(k_star + 1):
    a, b = swaps[i]
    A_cur = [b if x == a else x for x in A_cur]
    B_cur = [a if x == b else x for x in B_cur]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

c_before = [CATEGORY_PALETTE['red'] if n in A0    else CATEGORY_PALETTE['blue'] for n in G8.nodes()]
c_after  = [CATEGORY_PALETTE['red'] if n in A_cur else CATEGORY_PALETTE['blue'] for n in G8.nodes()]

draw_graph(G8, pos=pos8, ax=axes[0], node_color=c_before, node_size=650, font_size=9)
axes[0].set_title(f'BEFORE — cut = {cut_size_simple(G8, A0, B0)}')
draw_graph(G8, pos=pos8, ax=axes[1], node_color=c_after,  node_size=650, font_size=9)
axes[1].set_title(f'AFTER (prefix) — cut = {cut_size_simple(G8, A_cur, B_cur)}')

axes[2].plot(range(1, len(cumulative)+1), cumulative,
             marker='o', color=CATEGORY_PALETTE['blue'])
axes[2].axvline(k_star+1, color=CATEGORY_PALETTE['red'], ls='--', label=f'commit @ k*={k_star+1}')
style_axis(axes[2], title='Cumulative gain across the pass',
           xlabel='swap index $k$', ylabel='$G_k$', legend=True)
plt.show()

**Observation.** The cumulative gain is *unimodal* in this small example: it climbs to a peak and
then drops as additional forced swaps make things worse. Committing only the *prefix* up to the
peak is the essential trick that lets KL accept temporarily bad moves.

### 5.4 KL on the real datasets

In [ ]:
A_k, B_k = nx.community.kernighan_lin_bisection(G_karate, seed=RANDOM_SEED)
print(f'Karate  | |A|={len(A_k)}, |B|={len(B_k)}, cut size = {cut_size_simple(G_karate, A_k, B_k)}')

A_l, B_l = nx.community.kernighan_lin_bisection(G_lesmis, seed=RANDOM_SEED)
print(f'Les Mis | |A|={len(A_l)}, |B|={len(B_l)}, cut size = {cut_size_simple(G_lesmis, A_l, B_l)}')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_kl = [CATEGORY_PALETTE['red'] if n in A_k else CATEGORY_PALETTE['blue'] for n in G_karate.nodes()]
draw_graph(G_karate, pos=pos_karate, ax=axes[0], node_color=colors_kl, node_size=400, font_size=8)
axes[0].set_title(f'KL bisection — cut = {cut_size_simple(G_karate, A_k, B_k)}')

draw_graph(G_karate, pos=pos_karate, ax=axes[1], node_color=colors_truth, node_size=400, font_size=8)
axes[1].set_title('Ground truth (Mr. Hi vs. Officer)')
plt.show()

agree = sum(1 for n in G_karate.nodes() if (n in A_k) == (karate_truth[n] == 0))
print(f'Node-level agreement KL vs. ground truth: '
      f'{max(agree, G_karate.number_of_nodes()-agree)}/{G_karate.number_of_nodes()}')

**Observation.** KL forces **exactly equal** halves, so its 17/17 split cannot match the natural
16/18 split of the Karate Club. Even when KL “agrees” with the ground truth it does so for the
*wrong* reason — it was minimising cuts, not cohesion.

**Interpretation.** The histogram spreads over several integers — classic local-optimum behaviour.
In production, KL is often run many times from different seeds and the best cut is kept, or it is
used as a *refinement step* on top of another algorithm.

In [ ]:
# KL as a REFINEMENT of random balanced partitions
rng = np.random.default_rng(RANDOM_SEED)
nodes_list = list(G_karate.nodes())
refine_rows = []
for trial in range(20):
    perm = rng.permutation(nodes_list)
    half = len(perm) // 2
    A_rand, B_rand = list(perm[:half]), list(perm[half:])
    before = cut_size_simple(G_karate, A_rand, B_rand)
    A_ref, B_ref = nx.community.kernighan_lin_bisection(
        G_karate, partition=(set(A_rand), set(B_rand)), seed=int(trial))
    after = cut_size_simple(G_karate, A_ref, B_ref)
    refine_rows.append({'trial': trial, 'cut before': before, 'cut after': after,
                        'improvement': before - after})
df_ref = pd.DataFrame(refine_rows)
display(df_ref.describe().loc[['mean','min','max']])
print('Cases where KL improved the random partition:',
      int((df_ref["improvement"] > 0).sum()), 'out of', len(df_ref))

### 5.5 Take-home on KL

- KL minimises **cut size**, not *community quality*.
- The number of groups and their sizes are **inputs**, not outputs.
- It is a **local** greedy method whose answer depends on the initial bisection.
- Its natural niche is as a **post-processing refinement** of another algorithm's partition.
- *Clusters produced by graph partitioning are, in general, not communities* — the criterion
  ignores internal density and has no notion of *how many* communities are natural.

---
## 6. Hierarchical clustering with structural equivalence

### 6.1 From points to nodes — intuition

Classical hierarchical clustering groups *points* by their Euclidean distance. To apply the idea to
a network we need a *similarity* between *nodes*. The most natural network-based choice is
**structural equivalence**: two nodes are similar if they have similar *neighbourhoods*.

$$S^{SE}_{ij} \;=\; \frac{|\, N(i) \cap N(j)\,|}{|\, N(i) \cup N(j)\,|}$$

This is the **Jaccard similarity** of the neighbour sets: $1$ when the two nodes share *all*
neighbours, $0$ when they share *none*. Structurally equivalent nodes play the same role in the
network, even if they are not adjacent to each other.

### 6.2 Jaccard — tiny numeric example

Consider $N(i) = \{a,b,c,d\}$ and $N(j) = \{b,c,e\}$:

$$S^{SE}_{ij} = \frac{|\{b,c\}|}{|\{a,b,c,d,e\}|} = \frac{2}{5} = 0.4.$$

In [ ]:
def jaccard(A, B):
    A, B = set(A), set(B)
    if not (A | B):
        return 0.0
    return len(A & B) / len(A | B)

print('Jaccard({a,b,c,d}, {b,c,e}) =', jaccard('abcd', 'bce'))

### 6.3 Toy similarity heatmap

On the two-triangles-plus-bridge graph we already know which pairs should be similar: the two nodes
*inside* a triangle that are *not* the bridge node should be highly similar, because they share
most of their neighbours.

In [ ]:
def structural_equivalence_matrix(G):
    nodes = list(G.nodes())
    n = len(nodes)
    S = np.zeros((n, n))
    neigh = {v: set(G.neighbors(v)) for v in nodes}
    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if i == j:
                S[i, j] = 1.0
            else:
                union = neigh[u] | neigh[v]
                S[i, j] = len(neigh[u] & neigh[v]) / len(union) if union else 0.0
    return nodes, S


toy_nodes, toy_S = structural_equivalence_matrix(G_toy)
plot_heatmap(toy_S, labels=toy_nodes, title='Toy graph — structural equivalence heatmap',
             figure_size=(5.5, 4.5), fmt='{:.2f}')
plt.show()

**Reading.** The $3 \times 3$ block in the top-left corresponds to the left triangle; the one in the
bottom-right, to the right triangle. Within each block, most similarities are high — with one
exception: the *bridge endpoint* ($c$ or $d$) is slightly less similar to its triangle-mates
because it has the extra bridge neighbour.

### 6.4 From similarity to a dendrogram — linkage methods

Once we have a similarity (equivalently, a distance $d_{ij} = 1 - S_{ij}$), we agglomeratively merge
the two currently-closest clusters until a single cluster remains. The **linkage** rule decides how
to define *distance between clusters* from pairwise point distances:

| Linkage | Cluster distance | Tendency |
|---|---|---|
| **Single** | $\min_{i\in A,\,j\in B} d_{ij}$ | *Elongated, chain-like* clusters |
| **Complete** | $\max_{i\in A,\,j\in B} d_{ij}$ | *Compact, spherical* clusters |
| **Average** | $\text{mean}_{i\in A,\,j\in B} d_{ij}$ | *Intermediate*, robust |

Hierarchical clustering can be **agglomerative** (bottom-up merges, as above) or **divisive**
(top-down splits). *Girvan–Newman* in NB 13 is an example of the divisive family: it repeatedly
removes edges with highest betweenness.

### 6.5 Karate — similarity heatmap with community-ordered rows

In [ ]:
nodes_k, S_k = structural_equivalence_matrix(G_karate)

# Reorder rows so that Mr. Hi's faction comes first
order = sorted(range(len(nodes_k)), key=lambda i: karate_truth[nodes_k[i]])
S_reordered = S_k[np.ix_(order, order)]
node_order = [nodes_k[i] for i in order]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(S_k, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Similarity matrix — natural node order')
axes[0].set_xlabel('node'); axes[0].set_ylabel('node')
plt.colorbar(im0, ax=axes[0], label='$S_{ij}^{SE}$')

im1 = axes[1].imshow(S_reordered, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Reordered by ground-truth community')
axes[1].set_xticks(range(len(node_order))); axes[1].set_xticklabels(node_order, rotation=90, fontsize=6)
axes[1].set_yticks(range(len(node_order))); axes[1].set_yticklabels(node_order, fontsize=6)
# separator between the two factions
split = sum(1 for n in node_order if karate_truth[n] == 0)
axes[1].axhline(split - 0.5, color=CATEGORY_PALETTE['red'])
axes[1].axvline(split - 0.5, color=CATEGORY_PALETTE['red'])
plt.colorbar(im1, ax=axes[1], label='$S_{ij}^{SE}$')
plt.tight_layout(); plt.show()

**Interpretation.** The right panel reveals *block structure*: within-community similarities are
clearly above between-community similarities. The reason we sometimes miss this in the left panel
is merely that the nodes are not ordered — the statistical signal is there, hidden by the ordering.

### 6.6 Three dendrograms side by side — single / complete / average

In [ ]:
D_k = 1 - S_k
D_k = (D_k + D_k.T) / 2      # enforce symmetry
np.fill_diagonal(D_k, 0.0)   # enforce zero diagonal
condensed = squareform(D_k)  # validates automatically

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
linkages = {}
for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(condensed, method=method)
    linkages[method] = Z
    dendrogram(Z, labels=[str(n) for n in nodes_k], ax=ax,
               leaf_font_size=7, color_threshold=0.7*max(Z[:, 2]))
    ax.set_title(f'{method.capitalize()} linkage — Karate')
    ax.set_ylabel('distance $1-S^{SE}$')
plt.tight_layout(); plt.show()

**Reading a dendrogram.** Leaves at the bottom are individual nodes. Going upward, each *horizontal
line* marks a merge; its height is the cluster distance at merge time. A **horizontal cut** at any
height yields a *partition*: the higher the cut, the fewer and larger the clusters.

The three linkages agree on the "obvious" merges low in the tree but disagree near the top:

- **Single** links whichever clusters share the single most similar pair → chains tend to form.
- **Complete** demands that *all* pairs across clusters are similar → compact groups.
- **Average** is the compromise, usually the safest default.

### 6.7 Same dendrogram, two horizontal cuts — two partitions

In [ ]:
Z_avg = linkages['average']
heights = Z_avg[:, 2]
cut_low  = np.percentile(heights, 80)   # many small clusters
cut_high = np.percentile(heights, 95)   # few big clusters

labels_low  = fcluster(Z_avg, t=cut_low,  criterion='distance')
labels_high = fcluster(Z_avg, t=cut_high, criterion='distance')

def labels_to_colors(labels):
    pal = list(CATEGORY_PALETTE.values())
    return [pal[(int(l)-1) % len(pal)] for l in labels]


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
dendrogram(Z_avg, labels=[str(n) for n in nodes_k], ax=axes[0],
           leaf_font_size=6, color_threshold=cut_high)
axes[0].axhline(cut_low,  color=CATEGORY_PALETTE['red'],    ls='--', label=f'low  cut = {cut_low:.2f}')
axes[0].axhline(cut_high, color=CATEGORY_PALETTE['orange'], ls='--', label=f'high cut = {cut_high:.2f}')
axes[0].legend(); axes[0].set_title('Average-linkage dendrogram with two cuts')
axes[0].set_ylabel('distance $1-S^{SE}$')

draw_graph(G_karate, pos=pos_karate, ax=axes[1],
           node_color=labels_to_colors(labels_low),  node_size=350, font_size=7)
axes[1].set_title(f'Low cut — {len(set(labels_low))} clusters')

draw_graph(G_karate, pos=pos_karate, ax=axes[2],
           node_color=labels_to_colors(labels_high), node_size=350, font_size=7)
axes[2].set_title(f'High cut — {len(set(labels_high))} clusters')
plt.show()

### 6.8 Linkage comparison at a fixed number of clusters

Cut each of the three dendrograms at $k = 4$ clusters and compare: size distribution and the
*modularity* $Q$ of the induced partition. (Modularity is the central object of NB 13; we only
*use* it here as a scalar summary.)

In [ ]:
k_fixed = 4
rows = []
for method, Z in linkages.items():
    labels = fcluster(Z, t=k_fixed, criterion='maxclust')
    comms = [[nodes_k[i] for i, lab in enumerate(labels) if lab == c] for c in range(1, k_fixed+1)]
    Q = nx.community.modularity(G_karate, comms)
    rows.append({'linkage': method,
                 'cluster sizes': sorted(len(c) for c in comms),
                 'modularity Q': round(Q, 3)})
display(pd.DataFrame(rows))

**Interpretation.** Single linkage tends to produce one "giant" cluster and a handful of singletons
(a phenomenon known as *chaining*), which typically gives a poor modularity. Complete and average
yield more balanced partitions; average is usually a safe default.

### 6.9 Les Misérables dendrogram

To restate the three linkage names for clarity:
- **Single linkage** uses the *maximum* pairwise similarity (or *minimum* distance) between the two groups. It tolerates long thin chains of marginally-similar nodes — hence the elongated clusters and the "giant" effect you see above.
- **Complete linkage** uses the *minimum* pairwise similarity. To merge, the two groups must already be pairwise compatible — this enforces tight, spherical, similar-sized clusters.
- **Average linkage** uses the *mean* pairwise similarity — a compromise that behaves well on most real data.

Picking the right linkage is a modelling choice. For community detection in social / biological networks, **average** or **complete** are the safer defaults; **single** is useful when you expect literal chain structures (e.g. trophic food chains).


In [ ]:
nodes_l, S_l = structural_equivalence_matrix(G_lesmis)
D_l = 1 - S_l
D_l = (D_l + D_l.T) / 2
np.fill_diagonal(D_l, 0.0)
Z_l = linkage(squareform(D_l), method='average')

fig, ax = plt.subplots(figsize=(15, 5))
dendrogram(Z_l, labels=list(nodes_l), ax=ax, leaf_rotation=90, leaf_font_size=7,
           color_threshold=0.7*max(Z_l[:, 2]))
ax.set_title('Les Misérables — average-linkage dendrogram (structural equivalence)')
ax.set_ylabel('distance $1-S^{SE}$')
plt.tight_layout(); plt.show()

**Interpretation.** Characters that appear together in many chapters (the Bishop sub-plot, the
student revolutionaries, the Thénardier family, the Cosette/Valjean core) merge low in the
dendrogram, recovering Hugo's narrative arcs from co-occurrence alone.

---
## 7. Exercises

Each of the exercises below is self-contained. Scaffolding code is provided; the *interpretation*
is up to you.

### Exercise 1 — densest community in Les Misérables

Run Louvain on Les Misérables. Identify the community with **highest internal link density**. Is
it a **strong** community in the *stringent* sense?

*Why this matters.* A community with very high $\delta^{\text{int}}$ is a candidate for
a **cohesive block** — but on its own that is not enough to call it a *community* in the
strong sense (NB 12 §3): the internal density must *also* exceed the *external* density
of its nodes. This exercise trains you to combine the two diagnostics — density alone is
not a verdict.


In [ ]:
lesmis_parts = nx.community.louvain_communities(G_lesmis, seed=RANDOM_SEED)

rows = []
for i, C in enumerate(lesmis_parts):
    rows.append({'community': f'C{i}', '|C|': len(C),
                 'delta_int': round(internal_link_density(G_lesmis, C), 3),
                 'strong (stringent)': is_strong_community(G_lesmis, C),
                 'weak (stringent)':   is_weak_community(G_lesmis, C)})
df_ex1 = pd.DataFrame(rows).sort_values('delta_int', ascending=False).reset_index(drop=True)
display(df_ex1.head())

### Exercise 2 — Bell number $B_{30}$

Using the recurrence $B_{n+1} = \sum_{k=0}^{n} \binom{n}{k} B_k$, compute $B_{30}$ and compare it to
the number of atoms in the observable universe ($\sim 10^{80}$). How many $B_{30}$ partitions would
you have to inspect per second to enumerate them within the age of the universe ($\approx 4.3\times 10^{17}$ s)?

In [ ]:
B_list = bell_numbers(30)
print(f'B_30 = {B_list[30]:,}')
print(f'partitions per second to finish in the age of the universe: '
      f'{B_list[30] / 4.3e17:.3g}')

### Exercise 3 — Kernighan–Lin on the Florentine families

Apply KL bisection to `nx.florentine_families_graph()`. Does the resulting cut reproduce the
historical Medici vs. Strozzi divide?

In [ ]:
G_flor = nx.florentine_families_graph()
A_f, B_f = nx.community.kernighan_lin_bisection(G_flor, seed=RANDOM_SEED)
print('Partition A:', sorted(A_f))
print('Partition B:', sorted(B_f))
print('Cut size   :', cut_size_simple(G_flor, A_f, B_f))

pos_f = nx.spring_layout(G_flor, seed=RANDOM_SEED)
cols  = [CATEGORY_PALETTE['red'] if n in A_f else CATEGORY_PALETTE['blue'] for n in G_flor.nodes()]
plt.figure(figsize=(8, 5))
draw_graph(G_flor, pos=pos_f, node_color=cols, node_size=600, font_size=8)
plt.title('Florentine families — KL bisection')
plt.show()

### Exercise 4 — linkage shoot-out on Les Misérables

Run agglomerative hierarchical clustering on Les Misérables with all three linkage rules and cut
each dendrogram at $k = 6$. Which linkage produces the highest modularity?

In [ ]:
k_cut = 6
rows = []
for method in ['single', 'complete', 'average']:
    Z = linkage(squareform(D_l), method=method)
    labels = fcluster(Z, t=k_cut, criterion='maxclust')
    comms = [[nodes_l[i] for i, lab in enumerate(labels) if lab == c]
             for c in range(1, k_cut+1)]
    comms = [C for C in comms if C]  # drop empty
    Q = nx.community.modularity(G_lesmis, comms)
    rows.append({'linkage': method,
                 'n_clusters': len(comms),
                 'sizes': sorted(len(c) for c in comms),
                 'modularity Q': round(Q, 3)})
display(pd.DataFrame(rows))

### Exercise 5 (conceptual) — a weak community with a bad node

Suppose a node in $C$ has $k_v^{int} = 2$ and $k_v^{ext} = 5$. Can $C$ still be **weak** in the
stringent sense?

*Answer sketch.* Yes. The stringent weak condition says only that $\sum_v k_v^{int} > \sum_v k_v^{ext}$.
A single "bad" node that exports many external edges is *compensated* if the rest of $C$ has enough
internal links. Example: a clique of 6 nodes each with internal degree 5 and external degree 0
contributes $\sum k^{int} = 30$ against $\sum k^{ext} = 0$; adding one more node with $(k^{int},
k^{ext}) = (2,5)$ gives totals $32 > 5$. The community is *weak* but not *strong*.

---
## 8. Takeaway and bridge to NS 13 / NS 14

### What we now have in our toolbox

- A precise vocabulary for talking about communities: *cohesion*, *separation*, *internal/external
  degree*, *internal link density*, *spanning link*.
- Four formal definitions of community (strong/weak, stringent/less-stringent) and the scenarios
  that distinguish them.
- An appreciation of the partition space (Bell numbers) and of its richer cousins (*covers*,
  *hierarchies*), which justifies *every* heuristic method we will meet in NB 13 / NB 14.
- A working understanding of **Kernighan–Lin** bisection: its gain table, its locking mechanism,
  its sensitivity to initialisation, and the crucial observation that *graph partitioning is not
  community detection*.
- **Hierarchical clustering** end-to-end: Jaccard-based structural equivalence, linkage rules,
  dendrograms, horizontal cuts, modularity of each resulting partition.

### Which of these carry over to NB 13

- *Spanning links* and the hierarchical/divisive perspective feed directly into **Girvan–Newman**.
- *Internal link density* and *community degree* are the building blocks of **modularity** $Q$,
  the objective that **greedy Newman** and **Louvain** maximise.
- *Strong/weak (less-stringent)* is the property implicitly enforced by modularity-maximising
  algorithms: nodes tend to be placed in the community that captures the plurality of their edges.

### Which carry over to NB 14

- The *partition-space explosion* is the reason **label propagation** is appealing: it sidesteps
  enumeration entirely.
- Our concept-check on the Karate Club (what does "the right number" of communities even mean?)
  motivates the use of **NMI / ARI** against a ground truth and of **LFR benchmarks** as synthetic
  yardsticks.

Onwards to the algorithms.